# CV Screening Multi-Agent Demo

This notebook is a Colab/Kaggle launcher for the CV screening project. If the full project files are already present, it uses them directly. If only this notebook is opened in Colab, it automatically clones the GitHub repository, installs dependencies, and runs the deterministic CV screening pipeline.

Brief coverage: 2 specialist agents plus 1 orchestrator, trained PyTorch model tool, skill extraction tool, human-in-the-loop checkpoint, JSON logging, batch pipeline evaluation, and tests.

## 1. Runtime Check

This cell finds the project folder automatically. If the notebook is opened by itself in Colab, it clones the project from GitHub first.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def looks_like_project_root(path: Path) -> bool:
    return (path / "src" / "main.py").exists() and (path / "requirements.txt").exists()


GITHUB_REPO_URL = "https://github.com/cleora88/cv-screening-multiagent.git"

PROJECT_ROOT = Path.cwd()
if not looks_like_project_root(PROJECT_ROOT):
    candidates = [path for path in Path.cwd().rglob("requirements.txt") if looks_like_project_root(path.parent)]
    candidates += [path for path in Path("/content").glob("**/requirements.txt") if looks_like_project_root(path.parent)] if Path("/content").exists() else []
    candidates += [path for path in Path("/kaggle/input").glob("**/requirements.txt") if looks_like_project_root(path.parent)] if Path("/kaggle/input").exists() else []
    if candidates:
        PROJECT_ROOT = candidates[0].parent
    else:
        clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
        clone_dir = clone_parent / "cv-screening-multiagent"
        if not clone_dir.exists():
            print(f"Project files not found. Cloning repository from {GITHUB_REPO_URL}...")
            subprocess.run(["git", "clone", GITHUB_REPO_URL, str(clone_dir)], check=True)
        PROJECT_ROOT = clone_dir

os.chdir(PROJECT_ROOT)

required_paths = [
    PROJECT_ROOT / "src" / "main.py",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "data" / "sample_cvs.json",
    PROJECT_ROOT / "data" / "sample_jobs.json",
]

missing = [str(path.relative_to(PROJECT_ROOT)) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Notebook is not running from the project root, or files are missing: " + ", ".join(missing)
    )

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Files OK")

## 2. Install Dependencies

The deterministic run does not require Ollama. The project includes CrewAI dependencies, but this notebook runs the deterministic backend because it is the most reliable option in hosted notebooks.

In [ ]:
%pip install -q -r requirements.txt

## 3. Train Or Verify The Model

If `models/cv_fit_model.pt` is already included, this cell keeps it. If it is missing, it trains the baseline model using the existing project code.

In [ ]:
from pathlib import Path

model_path = Path("models/cv_fit_model.pt")
if model_path.exists():
    print(f"Model found: {model_path}")
else:
    print("Model not found. Training baseline model...")
    !python -m src.train_baseline

## 4. Run Single CV Screening

This uses the existing sample CV and job description from the `data` folder.

In [ ]:
!python -m src.main --runtime deterministic

## 5. Run Batch Screening

This screens every CV in `data/sample_cvs.json` against the selected sample job.

In [ ]:
!python -m src.main --runtime deterministic --batch-screening --cv-file data/sample_cvs.json --job-file data/sample_jobs.json

## 6. Pipeline Evaluation

This regenerates the end-to-end pipeline evaluation output with accuracy, class metrics, and a confusion matrix.

In [ ]:
!python -m src.evaluate_pipeline

## 7. Run Tests

This verifies the smoke tests, batch screening behavior, frontend helpers, and edge cases.

In [ ]:
!python -m pytest -q

## 8. View Generated Logs

The pipeline writes JSONL audit logs into the `logs` folder.

In [ ]:
from pathlib import Path

log_files = sorted(Path("logs").glob("*.jsonl"))
print(f"Found {len(log_files)} log file(s).")
for path in log_files[-5:]:
    print(path)